Step 1: Import necessary packages

In [1]:
import os
import sys
import numpy as np
import tkinter as tk
from tkinter import filedialog
import pandas as pd
import re
import scipy
import matplotlib.pyplot as plt
from packaging import version


Step 2: Set .py path and add to working path

In [ ]:
root = tk.Tk()
root.withdraw()
root.lift()
root.attributes("-topmost", 1)
current_path = filedialog.askdirectory() 

print (current_path)

os.chdir(current_path)
if current_path not in sys.path:
    sys.path.append(current_path)

import irm_unmix_functions as fc    

Step 3: Select files (single or multiple)

In [ ]:
files = fc.open_file()
    
file = files[0]

 Step 4: Read IRM data and plot

In [ ]:

D = fc.read_VSM3900_irm(file)
x = D["raw_data"]["field_irm"]
y = D["raw_data"]["remanence_irm"]
data = fc.interp_IRM(x, y, xmax=D["script"]["SCRIPT"]["Final field"]*1e+3)
data["smooth"] = fc.smooth_IRM(data["remanence"], n=11, polyorder=0)
data["gradient_raw"] = np.gradient(data["remanence"], data["field_log"])
data["gradient_smooth"] = np.gradient(data["smooth"], data["field_log"])

# plt.plot(data["field_log"], np.gradient(data["remanence"], data["field_log"]))
# plt.plot(data["field_log"], data["gradient"])

x = np.array(data["field_log"])
yr = np.array(np.maximum(data["gradient_raw"], 0))
ys = np.array(np.maximum(data["gradient_smooth"], 0))

if version.parse(np.__version__) < version.parse("2.0.0"):
    yr = yr/np.trapz(yr, x)
    ys = ys/np.trapz(ys, x)
else:
    yr = yr/np.trapezoid(yr, x)
    ys = ys/np.trapezoid(ys, x)

fig, ax = plt.subplots()
ax.scatter(x, yr, c="gray", s=20)    
ax.plot(x, ys, "b", linewidth=1)